# Baumkataster Analysis

Explores the Magdeburg city tree cadastre 2026 (`Baeume_SFM_2026.gpkg`):
species distribution, crown geometry, and allometric calibration of the height model.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import box

from src.config import AREAS, ALLOMETRIC_A, ALLOMETRIC_B

## Load Baumkataster

In [ ]:
BAUMKATASTER_PATH = Path("../references/Baeume_SFM_2026.gpkg")
bk = gpd.read_file(BAUMKATASTER_PATH)  # already EPSG:25832

# Columns:
#   Gattung lang        — species (Latin + German name)
#   Baumhoehe           — total height (m)
#   Kronendurchmesser   — crown diameter (m)
#   Stammumfang         — trunk circumference (cm)
#   Pflanzjahr          — planting year
#   Objektart lang      — location type (Öffentliches Grün / AMT 66 / Spielplatz)

print(f"Total trees (Magdeburg): {len(bk):,}")
print(f"CRS: {bk.crs}")
bk[["Gattung lang", "Baumhoehe", "Kronendurchmesser", "Stammumfang", "Pflanzjahr"]].describe()

## Species and location breakdown

In [ ]:
# Top species — citywide
print("=== TOP 20 SPECIES (Magdeburg citywide) ===")
print(bk["Gattung lang"].value_counts().head(20).to_string())

print()
print("=== LOCATION TYPE ===")
print(bk["Objektart lang"].value_counts().to_string())

## OVGU study area — crown diameter map

In [ ]:
from pyproj import Transformer

_to_utm = Transformer.from_crs("EPSG:4326", "EPSG:25832", always_xy=True)
ovgu = AREAS["ovgu_bbox"]
west_m, south_m = _to_utm.transform(ovgu["west"], ovgu["south"])
east_m, north_m = _to_utm.transform(ovgu["east"], ovgu["north"])
bbox_utm = gpd.GeoDataFrame(
    geometry=[box(west_m, south_m, east_m, north_m)],
    crs="EPSG:25832",
)

bk_ovgu = bk.clip(bbox_utm).copy()
bk_ovgu = bk_ovgu[bk_ovgu["Kronendurchmesser"] > 0].reset_index(drop=True)

print(f"Trees in OVGU bbox (with crown data): {len(bk_ovgu)}")
print(f"Crown diameter — mean: {bk_ovgu['Kronendurchmesser'].mean():.1f} m  "
      f"median: {bk_ovgu['Kronendurchmesser'].median():.1f} m  "
      f"max: {bk_ovgu['Kronendurchmesser'].max():.1f} m")

print("\n--- Species (OVGU) ---")
print(bk_ovgu["Gattung lang"].value_counts().head(15).to_string())

# Buffer each point by crown radius → actual circular footprint in map units
bk_crown = bk_ovgu.copy()
bk_crown["geometry"] = bk_crown.apply(
    lambda r: r.geometry.buffer(r["Kronendurchmesser"] / 2), axis=1
)

norm = mcolors.Normalize(vmin=bk_ovgu["Kronendurchmesser"].quantile(0.05),
                         vmax=bk_ovgu["Kronendurchmesser"].quantile(0.95))
cmap = cm.YlGn
colors = cmap(norm(bk_crown["Kronendurchmesser"].values))

fig, ax = plt.subplots(figsize=(10, 10))
bk_crown.plot(ax=ax, color=colors, edgecolor="darkgreen", linewidth=0.3, alpha=0.7)
bbox_utm.boundary.plot(ax=ax, color="red", linewidth=1.5, label="OVGU bbox")

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Crown diameter (m)", shrink=0.6)

ax.set_title(f"Baumkataster — OVGU area ({len(bk_crown)} trees, crown circles to scale)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend()
plt.tight_layout()
plt.show()

## Allometric calibration from Baumkataster

In [ ]:
# Fit ln(H) = A + B·ln(CPA) on Baumkataster (citywide, valid measurements only)
cal = bk[["Baumhoehe", "Kronendurchmesser"]].copy()
cal = cal[(cal["Baumhoehe"] > 1) & (cal["Kronendurchmesser"] > 0.5)]  # drop zeros/tiny
cal["CPA_m2"] = math.pi * (cal["Kronendurchmesser"] / 2) ** 2
cal["ln_H"]   = np.log(cal["Baumhoehe"])
cal["ln_CPA"] = np.log(cal["CPA_m2"])

# OLS: ln(H) = A + B·ln(CPA)
X = np.column_stack([np.ones(len(cal)), cal["ln_CPA"].values])
B_hat, resid, _, _ = np.linalg.lstsq(X, cal["ln_H"].values, rcond=None)
A_urban, B_urban = B_hat

r2 = 1 - np.sum((cal["ln_H"] - (A_urban + B_urban * cal["ln_CPA"])) ** 2) / \
         np.sum((cal["ln_H"] - cal["ln_H"].mean()) ** 2)

print("=== Baumkataster-calibrated allometric model (citywide) ===")
print(f"  H = exp({A_urban:.3f} + {B_urban:.3f} · ln(CPA))   R² = {r2:.3f}   n = {len(cal):,}")
print()
print("=== Current config (src/config.py) ===")
print(f"  H = exp({ALLOMETRIC_A:.3f} + {ALLOMETRIC_B:.3f} · ln(CPA))")
print()

# Compare predictions at representative crown diameters
print(f"{'Crown D (m)':>12}  {'CPA (m²)':>10}  {'H_urban (m)':>12}  {'H_config (m)':>13}  {'Bk median H':>12}")
for cd in [4, 6, 8, 10, 12, 14]:
    cpa = math.pi * (cd / 2) ** 2
    h_urban = math.exp(A_urban + B_urban * math.log(cpa))
    h_config = math.exp(ALLOMETRIC_A + ALLOMETRIC_B * math.log(cpa))
    bk_med = bk[(bk["Kronendurchmesser"].between(cd - 0.5, cd + 0.5)) &
                (bk["Baumhoehe"] > 1)]["Baumhoehe"].median()
    print(f"{cd:>12}  {cpa:>10.1f}  {h_urban:>12.1f}  {h_config:>13.1f}  {bk_med:>12.1f}")